In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from typing import List, Optional, Dict, Any

from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F

# ============================================================
# Helpers
# ============================================================

MONTH_ABB = {
    1: "jan", 2: "feb", 3: "mar", 4: "apr",
    5: "may", 6: "jun", 7: "jul", 8: "aug",
    9: "sep", 10: "oct", 11: "nov", 12: "dec"
}

# ============================================================
# Data Preparation
# ============================================================

def prepare_base_spark(
    sdf: DataFrame,
    id_col: str = "aID",
    month_col: str = "TIDPUNKT",
    adoption_col: str = "tariff_start",
    price_col: Optional[str] = "price",
    price_value: Optional[str] = None,
    price_values: Optional[List[str]] = None
) -> DataFrame:
    """
    支援兩種模式：

    1. 單一 price:
       price_values=["high"]
       -> 欄位維持原本名字：
          top3_mean_consumption, mean_consumption, ...

    2. 多個 price:
       price_values=["high", "all"]
       -> 轉成 wide:
          top3_mean_consumption_high, top3_mean_consumption_all, ...
    """

    out = sdf

    # backward compatibility
    if price_values is None:
        if price_value is not None:
            price_values = [price_value]
        else:
            price_values = None

    # ==============================
    # no price column or no filtering
    # ==============================
    if price_col is None or price_col not in sdf.columns or price_values is None:
        out = (
            out
            .withColumn("id", F.col(id_col).cast("string"))
            .withColumn("month", F.to_date(F.col(month_col)))
            .withColumn("adoption_month", F.to_date(F.col(adoption_col)))
            .withColumn("top3_mean_consumption", F.col("top3_mean_consumption").cast("double"))
            .withColumn("mean_consumption", F.col("mean_consumption").cast("double"))
            .withColumn("variance_consumption", F.col("variance_consumption").cast("double"))
            .withColumn("total_consumption", F.col("total_consumption").cast("double"))
            .select(
                "id",
                "month",
                "adoption_month",
                "top3_mean_consumption",
                "mean_consumption",
                "variance_consumption",
                "total_consumption"
            )
            .filter(F.col("id").isNotNull())
            .filter(F.col("month").isNotNull())
        )
        return out

    # ==============================
    # single price: 原本邏輯
    # ==============================
    if len(price_values) == 1:
        pv = price_values[0]

        out = (
            out
            .filter(F.col(price_col) == F.lit(pv))
            .withColumn("id", F.col(id_col).cast("string"))
            .withColumn("month", F.to_date(F.col(month_col)))
            .withColumn("adoption_month", F.to_date(F.col(adoption_col)))
            .withColumn("top3_mean_consumption", F.col("top3_mean_consumption").cast("double"))
            .withColumn("mean_consumption", F.col("mean_consumption").cast("double"))
            .withColumn("variance_consumption", F.col("variance_consumption").cast("double"))
            .withColumn("total_consumption", F.col("total_consumption").cast("double"))
            .select(
                "id",
                "month",
                "adoption_month",
                "top3_mean_consumption",
                "mean_consumption",
                "variance_consumption",
                "total_consumption"
            )
            .filter(F.col("id").isNotNull())
            .filter(F.col("month").isNotNull())
        )

        return out

    # ==============================
    # multiple prices: wide format
    # ==============================
    base = (
        out
        .filter(F.col(price_col).isin(price_values))
        .withColumn("id", F.col(id_col).cast("string"))
        .withColumn("month", F.to_date(F.col(month_col)))
        .withColumn("adoption_month", F.to_date(F.col(adoption_col)))
        .withColumn("price_key", F.col(price_col).cast("string"))
        .withColumn("top3_mean_consumption", F.col("top3_mean_consumption").cast("double"))
        .withColumn("mean_consumption", F.col("mean_consumption").cast("double"))
        .withColumn("variance_consumption", F.col("variance_consumption").cast("double"))
        .withColumn("total_consumption", F.col("total_consumption").cast("double"))
        .filter(F.col("id").isNotNull())
        .filter(F.col("month").isNotNull())
    )

    agg_exprs = [
        F.min("adoption_month").alias("adoption_month")
    ]

    value_cols = [
        "top3_mean_consumption",
        "mean_consumption",
        "variance_consumption",
        "total_consumption"
    ]

    for pv in price_values:
        suffix = pv.replace(" ", "_").replace("-", "_")

        for c in value_cols:
            agg_exprs.append(
                F.first(
                    F.when(F.col("price_key") == F.lit(pv), F.col(c)),
                    ignorenulls=True
                ).alias(f"{c}_{suffix}")
            )

    wide = (
        base
        .groupBy("id", "month")
        .agg(*agg_exprs)
    )

    return wide


# ============================================================
# Cohort / Risk-set construction
# ============================================================

def get_distinct_ti(
    base_df: DataFrame,
    min_ti: Optional[str] = None,
    max_ti: Optional[str] = None
) -> DataFrame:
    """
    取得所有 adoption cohort 時點 Ti。
    min_ti / max_ti 可用來限制 cohort 範圍，避免資料膨脹。
    """
    ti = (
        base_df
        .select(F.col("adoption_month").alias("Ti"))
        .where(F.col("Ti").isNotNull())
        .distinct()
    )

    if min_ti is not None:
        ti = ti.where(F.col("Ti") >= F.lit(min_ti))
    if max_ti is not None:
        ti = ti.where(F.col("Ti") <= F.lit(max_ti))

    return ti


def get_user_status(base_df: DataFrame) -> DataFrame:
    """
    每位 user 對應一個 adoption_month。
    若資料中同 id 有多個 adoption_month，取最早非空值。
    """
    return (
        base_df
        .groupBy("id")
        .agg(F.min("adoption_month").alias("adoption_month"))
    )


def build_risk_set_rows(
    base_df: DataFrame,
    lookback_months: int = 12,
    min_ti: Optional[str] = None,
    max_ti: Optional[str] = None,
    match_months: Optional[List[int]] = None,
    control_type: str = "never_treated"
) -> DataFrame:

    ti_df = get_distinct_ti(base_df, min_ti=min_ti, max_ti=max_ti)
    user_status = get_user_status(base_df)

    ti_bounds = ti_df.agg(
        F.min("Ti").alias("min_Ti"),
        F.max("Ti").alias("max_Ti")
    ).first()

    if ti_bounds["min_Ti"] is None or ti_bounds["max_Ti"] is None:
        spark = SparkSession.builder.getOrCreate()
        return spark.createDataFrame([], schema="""
            id string,
            Ti date,
            adoption_month date,
            group string,
            month date
        """)

    global_min_month = F.add_months(F.lit(ti_bounds["min_Ti"]), -lookback_months)
    global_max_month = F.lit(ti_bounds["max_Ti"])

    monthly = (
        base_df
        .where(F.col("month") >= global_min_month)
        .where(F.col("month") < global_max_month)
        .alias("m")
    )

    if control_type == "risk_set":
        cond = (
            (F.col("u.adoption_month") == F.col("t.Ti")) |
            (F.col("u.adoption_month").isNull()) |
            (F.col("u.adoption_month") > F.col("t.Ti"))
        )
    elif control_type == "never_treated":
        cond = (
            (F.col("u.adoption_month") == F.col("t.Ti")) |
            (F.col("u.adoption_month").isNull())
        )
    else:
        raise ValueError("control_type must be 'risk_set' or 'never_treated'")

    base_value_cols = [
        c for c in base_df.columns
        if c not in ["id", "month", "adoption_month"]
    ]

    risk_rows = (
        monthly
        .crossJoin(F.broadcast(ti_df).alias("t"))
        .join(user_status.alias("u"), on="id", how="inner")
        .where(
            (F.col("m.month") < F.col("t.Ti")) &
            (F.col("m.month") >= F.add_months(F.col("t.Ti"), -lookback_months)) &
            cond
        )
        .withColumn(
            "group",
            F.when(F.col("u.adoption_month") == F.col("t.Ti"), F.lit("treated"))
             .otherwise(F.lit("control"))
        )
        .select(
            F.col("id"),
            F.col("t.Ti").alias("Ti"),
            F.col("u.adoption_month").alias("adoption_month"),
            F.col("group"),
            F.col("m.month").alias("month"),
            *[F.col(f"m.{c}").alias(c) for c in base_value_cols]
        )
    )

    if match_months is not None:
        risk_rows = risk_rows.where(F.month(F.col("month")).isin(match_months))

    return risk_rows



def prepare_matching_data(
    sdf: DataFrame,
    id_col: str = "aID",
    month_col: str = "TIDPUNKT",
    adoption_col: str = "tariff_start",
    price_col: Optional[str] = "price",
    price_value: Optional[str] = "all",
    price_values: Optional[List[str]] = None,
    lookback_months: int = 12,
    min_ti: Optional[str] = None,
    max_ti: Optional[str] = None,
    match_months: Optional[List[int]] = None,
    control_type: str = "never_treated",
    repartition_by_ti: bool = True,
    verbose: bool = True
) -> DataFrame:

    if verbose:
        print("Preparing base data...")

    base = prepare_base_spark(
        sdf,
        id_col=id_col,
        month_col=month_col,
        adoption_col=adoption_col,
        price_col=price_col,
        price_value=price_value,
        price_values=price_values
    )

    if verbose:
        print("Building risk set...")

    risk_rows = build_risk_set_rows(
        base,
        lookback_months=lookback_months,
        min_ti=min_ti,
        max_ti=max_ti,
        match_months=match_months,
        control_type=control_type
    )

    if repartition_by_ti:
        risk_rows = risk_rows.repartition("Ti")

    risk_rows = risk_rows.cache()

    if verbose:
        print("risk_rows count =", risk_rows.count())

    return risk_rows